# Short demo — Prompt engineering in practice
## From a complaint text to a structured label, one prompt version at a time

*AI Architecture — Day 1, Module 4 (about 25 minutes, before Demo 1)*

### Read this first: assumptions and simplifications

| Topic | What we assume in this demo | Why it matters |
|---|---|---|
| **The task** | Read the opening lines of a customer complaint and return two labels: the **topic** (Fees, Fraud, Service quality, Mortgage, Investments, Other) and the **tone level** (0 calm or positive, 1 neutral, 2 annoyed, 3 angry or threatening escalation). The tone is the feature that Demo 1 uses as `sentiment_score`; here it is an ordinal level, the more robust form of the same information. | Two labels of different nature: one is a category, the other an intensity. |
| **The data** | `complaint_eval_set.csv`: 45 synthetic complaints with labels produced by the same rules that generated the texts — 5 marked `example` (few-shot examples) and 40 marked `test` (never shown to the prompt). In a real project the labels come from complaint handlers, with a written rule book. | We score prompts against labels; without labels there is no evaluation, only opinions. |
| **The LLM** | OpenAI API, `openai` package. The key is read from a local `.env` file that contains **only the key** (no `OPENAI_API_KEY=` in front), searched in the notebook folder and in the folders above it. The `.env` file always wins: any `OPENAI_API_KEY` environment variable left on the computer (for example an old, revoked key) is ignored when the file exists, and is used only as a fallback when there is no `.env`. The `.env` file must never be shared or committed (add it to `.gitignore`). Two models: a small one (`gpt-4o-mini`) and a larger one (`gpt-4.1`); replace them with any two current models. Prices are typed in by hand from the vendor's price list and are **illustrative**. | Without the key the cells that call the API are skipped and print a notice; the demo needs the key to be meaningful. |
| **Cache** | Every answer is stored in `llm_cache.json`, so re-running a cell costs nothing and the trainer can warm the cache before the session. Delete the file to start again. | Live demos should not wait for 40 API calls twice. |
| **Metrics** | Accuracy on the topic, accuracy on the tone level (exact, and within ±1), share of invalid answers, cost per 1,000 complaints, median latency. | The four numbers that a prompt change must be judged on together. |
| **Scope** | This is the mechanism of prompt versioning and evaluation, not a complete study: 40 test complaints give rough estimates (one complaint is 2.5 points of accuracy). | Module 5 adds the metrics for open-ended answers and LLM-as-a-judge. |

### The storyline

1. A naive prompt, and why its answers cannot be scored
2. A structured prompt: role, definitions, scale, delimiters, JSON — scored on the 40 test complaints
3. A few-shot version: five examples inside the prompt — scored again
4. Two models on the same prompt: quality, cost and latency together
5. Non-determinism: the same complaint, five times
6. Prompts as assets: the record we keep for each version

In [ ]:
import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 120)

MODEL_A = "gpt-4o-mini"          # small and cheap
MODEL_B = "gpt-4.1"              # larger; replace both with any two current models
PRICES = {MODEL_A: (0.15, 0.60), MODEL_B: (2.00, 8.00)}   # USD per million tokens (input, output) — ILLUSTRATIVE, check the price list

def load_key_from_env_file(filename=".env"):
    """Return (key, path) from a .env file that holds only the key, looking in this folder and the folders above it."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / filename
        if env_file.is_file():
            for line in env_file.read_text(encoding="utf-8-sig").splitlines():   # utf-8-sig drops the hidden BOM that Notepad may add
                line = line.strip()
                if line and not line.startswith("#"):
                    key = line.split("=", 1)[-1].strip().strip('"').strip("'")   # also accepts KEY=value or a quoted key
                    return key, env_file
    return None, None

# The .env file always wins; an OPENAI_API_KEY variable on the computer is only a fallback.
API_KEY, source = load_key_from_env_file()
if API_KEY:
    os.environ["OPENAI_API_KEY"] = API_KEY               # overwrite any old key for this session
    print(f"OpenAI key read from {source} (ends with ...{API_KEY[-4:]})")   # never print the whole key
else:
    API_KEY = os.environ.get("OPENAI_API_KEY")
    if API_KEY:
        print(f"No .env file found: using the OPENAI_API_KEY environment variable (ends with ...{API_KEY[-4:]})")

try:
    from openai import OpenAI                       # pip install openai
    RUN_LLM = bool(API_KEY)
except ImportError:
    RUN_LLM = False

if RUN_LLM:
    client = OpenAI(api_key=API_KEY)                # the key is passed explicitly: no ambiguity about which one is used
    print("LLM client ready:", MODEL_A, "and", MODEL_B)
else:
    print("openai package or API key (.env file or OPENAI_API_KEY) not found: the cells that call the API will be skipped.")


One helper function for the whole notebook: it sends a system prompt and a complaint text to a model, asks
for JSON when we want a structured answer, measures the time and the tokens, and keeps every answer in a
cache file so that the same question is never paid for twice.

In [2]:
CACHE_FILE = "llm_cache.json"
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def call_llm(model, system_prompt, text, json_mode=True, temperature=0, run=0):
    """Return a dict: answer (parsed JSON or raw text), input_tokens, output_tokens, seconds."""
    key = hashlib.md5(f"{model}|{system_prompt}|{text}|{json_mode}|{temperature}|{run}".encode()).hexdigest()
    if key in cache:
        return cache[key]
    messages = ([{"role": "system", "content": system_prompt}] if system_prompt else []) + \
               [{"role": "user", "content": "Complaint:\n<<<\n" + text + "\n>>>"}]
    options = {"model": model, "messages": messages, "temperature": temperature}
    if json_mode:
        options["response_format"] = {"type": "json_object"}
    start = time.perf_counter()
    response = client.chat.completions.create(**options)
    seconds = time.perf_counter() - start
    content = response.choices[0].message.content
    try:
        answer = json.loads(content) if json_mode else content
    except json.JSONDecodeError:
        answer = None                                                  # invalid JSON: counted as an invalid answer
    result = {"answer": answer, "input_tokens": response.usage.prompt_tokens,
              "output_tokens": response.usage.completion_tokens, "seconds": round(seconds, 2)}
    cache[key] = result
    json.dump(cache, open(CACHE_FILE, "w"))
    return result

## 1. The labelled complaints

Five complaints are reserved as examples for the few-shot prompt; forty are the test set. The labels follow a
rule book that we will copy into the prompt: a prompt and its evaluation must speak the same language.

In [3]:
eval_set = pd.read_csv("complaint_eval_set.csv")
examples = eval_set[eval_set["split"] == "example"].reset_index(drop=True)
test = eval_set[eval_set["split"] == "test"].reset_index(drop=True)

print("examples:", len(examples), "  test:", len(test))
print(test["topic"].value_counts().to_dict())
print("tone levels:", test["tone_level"].value_counts().sort_index().to_dict())
test[["complaint_id", "product", "complaint_excerpt", "topic", "tone_level"]].head(5)

examples: 5   test: 40
{'Fees': 14, 'Mortgage': 8, 'Service quality': 7, 'Fraud': 6, 'Investments': 3, 'Other': 2}
tone levels: {0: 10, 1: 10, 2: 10, 3: 10}


,complaint_id,product,complaint_excerpt,topic,tone_level
0,CMP-002525,Mortgage,An insurance premium of 683 euros was added to my mortgage instalment without my consent. Thank you in advance for l...,Mortgage,0
1,CMP-001436,Current account,I was charged 40 euros in fees that were never mentioned when I opened the account. Thank you in advance for looking...,Fees,0
2,CMP-000664,Credit card,My card was charged twice for the same purchase of 56 euros. Your staff have been helpful so far; I just need this l...,Fees,0
3,CMP-001482,Investments,My advisor moved my savings into a fund I never approved and the position is now 917 euros below what I invested. I ...,Investments,0
4,CMP-002869,Credit card,The cash-advance fee of 153 euros on my card is not the one stated in the terms. Thank you in advance for looking in...,Fees,0


## 2. Version 1: the naive prompt

This is how most people write their first prompt. We send it for three complaints and read the answers.

In [4]:
PROMPT_V1 = "Classify this complaint."      # that is the whole instruction; the complaint follows

if RUN_LLM:
    for text in test["complaint_excerpt"].head(3):
        result = call_llm(MODEL_A, PROMPT_V1, text, json_mode=False)
        print(result["answer"], "\n" + "-" * 80)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************ZwkA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

> **Why this cannot be scored.** The answers are prose; the categories are invented by the model, they
> differ from one complaint to the next, and there is no field a program can read. To measure anything we
> need answers that always have the same shape, from a list of labels we chose.

## 3. Version 2: the structured prompt

Role, the definitions of the six topics (the same rule book as the labels), a four-level tone scale, a rule
for the ambiguous case, delimited input, JSON output.

In [ ]:
PROMPT_V2 = """You are a complaints analyst at a retail bank.
Read the customer complaint and return a JSON object with two fields.

"topic": exactly one of
- "Fees": charges, fees, instalments or amounts the customer disputes but does not deny having caused
- "Fraud": transactions or debits the customer says they never made or authorised
- "Service quality": delays, blocked accounts or cards, unanswered requests, operational errors
- "Mortgage": any complaint about a mortgage
- "Investments": any complaint about investments or an advisor
- "Other": anything else, including insurance claims

"tone_level": an integer from 0 to 3
- 0: calm or positive
- 1: neutral, factual
- 2: annoyed or disappointed
- 3: angry, or threatening to escalate (ombudsman, legal action, press)

If the complaint fits two topics, choose the one that describes the money at stake.
Answer with the JSON object only."""

TOPICS = ["Fees", "Fraud", "Service quality", "Mortgage", "Investments", "Other"]

The evaluation is the same for every version, so we write it once: run the prompt on the 40 test complaints,
compare with the labels, and compute the five numbers.

In [ ]:
def evaluate(model, system_prompt, version_name):
    """Run a prompt on the test set and return one row of metrics plus the detailed answers."""
    rows = []
    for _, row in test.iterrows():
        result = call_llm(model, system_prompt, row["complaint_excerpt"])
        answer = result["answer"] if isinstance(result["answer"], dict) else {}
        rows.append({"complaint_id": row["complaint_id"], "topic": row["topic"], "tone_level": row["tone_level"],
                     "pred_topic": answer.get("topic"), "pred_tone": answer.get("tone_level"),
                     "input_tokens": result["input_tokens"], "output_tokens": result["output_tokens"], "seconds": result["seconds"]})
    detail = pd.DataFrame(rows)
    valid = detail["pred_topic"].isin(TOPICS) & detail["pred_tone"].isin([0, 1, 2, 3])
    price_in, price_out = PRICES[model]
    cost_per_1000 = (detail["input_tokens"].mean() * price_in + detail["output_tokens"].mean() * price_out) / 1e6 * 1000
    metrics = {"version": version_name, "model": model,
               "topic accuracy": (detail["pred_topic"] == detail["topic"]).mean(),
               "tone accuracy": (detail["pred_tone"] == detail["tone_level"]).mean(),
               "tone within ±1": ((detail["pred_tone"] - detail["tone_level"]).abs() <= 1).mean(),
               "invalid answers": 1 - valid.mean(),
               "cost per 1,000 (USD)": cost_per_1000,
               "median latency (s)": detail["seconds"].median()}
    return metrics, detail

results = []
if RUN_LLM:
    metrics_v2, detail_v2 = evaluate(MODEL_A, PROMPT_V2, "v2 structured")
    results.append(metrics_v2)
pd.DataFrame(results).round(3)

Where does the structured prompt still fail? The confusion between topics tells us which definitions need
sharpening — and which labels were debatable in the first place.

In [ ]:
if RUN_LLM:
    print(pd.crosstab(detail_v2["topic"], detail_v2["pred_topic"], rownames=["label"], colnames=["predicted"]))
    print()
    print(pd.crosstab(detail_v2["tone_level"], detail_v2["pred_tone"], rownames=["label"], colnames=["predicted"]))

## 4. Version 3: few-shot — five examples inside the prompt

Same instructions, plus five complaints with their labels, one per tone level and one fraud case. The examples
fix what the definitions leave open: the granularity of the topics, the boundary between "annoyed" and
"angry", the exact output shape.

In [ ]:
example_block = "\n\nExamples:\n" + "\n".join(
    f'Complaint: "{row["complaint_excerpt"]}"\nAnswer: {{"topic": "{row["topic"]}", "tone_level": {row["tone_level"]}}}'
    for _, row in examples.iterrows())

PROMPT_V3 = PROMPT_V2 + example_block
print(PROMPT_V3[-900:])           # the tail of the prompt: the examples

In [ ]:
if RUN_LLM:
    metrics_v3, detail_v3 = evaluate(MODEL_A, PROMPT_V3, "v3 few-shot")
    results.append(metrics_v3)
pd.DataFrame(results).round(3)

> **What usually happens.** The few-shot version gains a few points on the tone level, where the scale is a
> matter of judgement, and costs more input tokens on every call: the examples are sent 1,000 times for 1,000
> complaints. Whether the gain is worth the cost is a business decision, and it is why cost sits in the same
> table as accuracy.

## 5. Two models on the same prompt

The larger model, same prompt, same test set. Three columns to read together: accuracy, cost, latency.

In [ ]:
if RUN_LLM:
    metrics_v3_b, detail_v3_b = evaluate(MODEL_B, PROMPT_V3, "v3 few-shot")
    results.append(metrics_v3_b)
pd.DataFrame(results).round(3)

> **Business reading.** A better prompt usually buys more than a bigger model, and the small model with a good
> prompt is often the reasonable choice: a few points of accuracy against a multiple of the cost and the
> latency. The decision, with the numbers behind it, goes on the Decision Card under "options considered".

## 6. Non-determinism: the same complaint, five times

At temperature 0 the answers should be almost always identical; at temperature 1 they vary. For
classification and extraction we keep the temperature low — and we still test on a set, not on one answer.

In [ ]:
if RUN_LLM:
    text = test.loc[0, "complaint_excerpt"]
    for temperature in [0, 1]:
        answers = [json.dumps(call_llm(MODEL_A, PROMPT_V2, text, temperature=temperature, run=i)["answer"]) for i in range(5)]
        print(f"temperature {temperature}: {len(set(answers))} distinct answer(s) out of 5")
        for a in sorted(set(answers)):
            print("   ", a)

## 7. Prompts as assets: the record we keep

Three versions of the same prompt now exist. In a project they live in a repository or a prompt registry, each
with a version identifier, the model it was tested with, the test set and the scores. The table below is that
record; the version identifier is what appears in every log line of the production system, so that a change
in the answers can be traced to a change in the prompt or in the model.

In [ ]:
PROMPTS = {"complaint-labels v1 (naive)": PROMPT_V1,
           "complaint-labels v2 (structured)": PROMPT_V2,
           "complaint-labels v3 (few-shot)": PROMPT_V3}
print({name: f"{len(p.split())} words" for name, p in PROMPTS.items()})

record = pd.DataFrame(results).round(3)
record.insert(0, "test set", f"complaint_eval_set.csv ({len(test)} complaints)")
record.insert(0, "date", pd.Timestamp.today().strftime("%Y-%m-%d"))
record

## 8. What to take away

- A prompt is a specification: definitions, scale, format and the rule for the ambiguous case turn prose
  answers into a measurable component.
- The labels and the prompt must share the same rule book; most "model errors" are disagreements between the
  two.
- Every change — wording, examples, model — is a release: it is scored on the same test set, and the record is
  kept.
- The tone level produced here is the `sentiment_score` of Demo 1, seen from the other side: there it is a
  feature, here it is an output to be trusted only after this kind of test.